# Audio Recommendation Engine (BEATs Embeddings) — SETUP / ARCHIVE

**Kernel crashes?** Do **not** extract embeddings in Jupyter.

Use this workflow instead:

1. Terminal: `python create_embeddings.py` (resumable; small batches; safe to re-run)
2. Open **`recommend.ipynb`** to query recommendations (loads `embeddings.npy` only)

Artifacts already present: `embeddings.npy`, `id_map.json`, `nn_index.faiss`.
You can skip straight to `recommend.ipynb`.

In [9]:
# Cell 1: Install dependencies
!pip install pandas requests torch torchaudio numpy scikit-learn tqdm faiss-cpu miniaudio --quiet

In [10]:
import json
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import torch
from tqdm.notebook import tqdm

try:
    import faiss
    USE_FAISS = True
    print("Using FAISS for similarity search.")
except ImportError:
    from sklearn.neighbors import NearestNeighbors
    USE_FAISS = False
    print("FAISS not available, falling back to sklearn NearestNeighbors.")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("WARNING: No GPU detected. Running on CPU (embedding extraction will be slow).")

Using FAISS for similarity search.


In [11]:
# Cell 2: Configuration

CSV_PATH = "audio_speech_labels.csv"
AUDIO_URL_COLUMN = "streamableUrl"
AUDIO_DIR = Path("audio_cache")
EMBEDDINGS_PATH = Path("embeddings.npy")
ID_MAP_PATH = Path("id_map.json")
INDEX_PATH = Path("nn_index.pkl")

BATCH_SIZE = 16
TARGET_SR = 16000
MAX_AUDIO_LENGTH_SEC = 30
DOWNLOAD_WORKERS = 8
DOWNLOAD_TIMEOUT = 60

AUDIO_DIR.mkdir(exist_ok=True)

In [12]:
# Cell 3: Data Ingestion

df = pd.read_csv(CSV_PATH)
print(f"Raw dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

# Filter to non_speech, completed rows with valid URLs
df = df[
    (df["speech_label"] == "non_speech")
    & (df["processing_status"] == "completed")
    & (df[AUDIO_URL_COLUMN].notna())
    & (df[AUDIO_URL_COLUMN].str.strip() != "")
].reset_index(drop=True)

print(f"Filtered dataset: {df.shape[0]} rows")
df[["id", AUDIO_URL_COLUMN, "speech_label", "duration"]].head()

Raw dataset: 6451 rows, 16 columns
Columns: ['id', 'createdAt', 'updatedAt', 'uploadedUrl', 'streamableUrl', 'transcodeJobId', 'transcodeStatus', 'transcodeMessage', 'duration', 'speech_label', 'speech_ratio', 'speech_duration_seconds', 'total_duration_seconds', 'speech_segment_count', 'processing_status', 'processing_error']
Filtered dataset: 1885 rows


,id,streamableUrl,speech_label,duration
0,3a7a8ebd-5b75-4c20-89cb-bd9906d8591b,https://says-api-streamable-audio-dev.s3.amazo...,non_speech,NaN
1,f6ad8d62-e425-4e99-bd61-ec460691dfc6,https://says-api-streamable-audio-dev.s3.amazo...,non_speech,NaN
2,1860aaba-71e2-49cd-8f0f-0ce1807e21c2,https://says-api-streamable-audio-dev.s3.amazo...,non_speech,NaN
3,6e7fd4fd-d72a-4fcd-a29a-02f7c9267278,https://says-api-streamable-audio-dev.s3.amazo...,non_speech,NaN
4,a4ba958a-994b-43ca-bae2-a60cd84ad3d3,https://says-api-streamable-audio-dev.s3.amazo...,non_speech,NaN


In [13]:
# Cell 4: Audio Download (Multithreaded)

def download_audio(row):
    """Download a single audio file. Returns (id, filepath) or (id, None) on failure."""
    clip_id = row["id"]
    url = row[AUDIO_URL_COLUMN].strip()
    ext = ".mp3" if ".mp3" in url else ".wav" if ".wav" in url else ".mp3"
    filepath = AUDIO_DIR / f"{clip_id}{ext}"

    if filepath.exists():
        return clip_id, str(filepath), None

    try:
        resp = requests.get(url, stream=True, timeout=DOWNLOAD_TIMEOUT)
        resp.raise_for_status()
        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        return clip_id, str(filepath), None
    except Exception as e:
        return clip_id, None, f"{type(e).__name__}: {e}"


download_errors = []
downloaded_files = {}  # id -> filepath

rows_to_download = df.to_dict("records")

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_audio, row): row["id"] for row in rows_to_download}

    with tqdm(total=len(futures), desc="Downloading audio") as pbar:
        for future in as_completed(futures):
            clip_id, filepath, error = future.result()
            if error:
                download_errors.append({"id": clip_id, "error": error})
            else:
                downloaded_files[clip_id] = filepath
            pbar.update(1)

print(f"\nDownloaded/cached: {len(downloaded_files)} | Failed: {len(download_errors)}")
if download_errors:
    print(f"First 5 errors: {download_errors[:5]}")


Downloaded/cached: 1885 | Failed: 0


In [14]:
# Cell 5: Audio Validation

import miniaudio


def load_audio_safe(filepath, target_sr=TARGET_SR):
    """Decode audio with miniaudio (works for mp3 without ffmpeg)."""
    decoded = miniaudio.decode_file(
        str(filepath),
        nchannels=1,
        sample_rate=target_sr,
    )
    if not decoded.samples:
        raise ValueError("Empty waveform")

    waveform = torch.tensor(decoded.samples, dtype=torch.float32).unsqueeze(0)
    return waveform, decoded.sample_rate


valid_clips = []  # list of (id, filepath)
validation_errors = []

for clip_id, filepath in tqdm(downloaded_files.items(), desc="Validating audio"):
    try:
        load_audio_safe(filepath)
        valid_clips.append((clip_id, filepath))
    except Exception as e:
        validation_errors.append({"id": clip_id, "error": str(e)})

print(f"Valid clips: {len(valid_clips)} | Validation failures: {len(validation_errors)}")
if validation_errors:
    print("First 3 errors for debugging:")
    for err in validation_errors[:3]:
        print(f"  {err['id']}: {err['error']}")

Validating audio:   0%|          | 0/1885 [00:00<?, ?it/s]

Valid clips: 1885 | Validation failures: 0


## BEATs Model Setup

The next cell uses `beats_setup.py` to automatically download the BEATs source files and checkpoint on first run.

In [15]:
# Cell 6: Load BEATs Model

from beats_setup import load_beats_model

beats_model = load_beats_model(DEVICE)
print(f"BEATs model loaded on {DEVICE}")

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATs model loaded on cpu


In [16]:
# Cell 7: Dataset and DataLoader

class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, clips, target_sr=TARGET_SR, max_len_sec=MAX_AUDIO_LENGTH_SEC):
        self.clips = clips  # list of (id, filepath)
        self.target_sr = target_sr
        self.max_samples = target_sr * max_len_sec

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_id, filepath = self.clips[idx]
        waveform, sr = load_audio_safe(filepath, target_sr=self.target_sr)

        waveform = waveform.squeeze(0)  # (samples,)

        # Pad or truncate
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[: self.max_samples]
        elif waveform.shape[0] < self.max_samples:
            pad_len = self.max_samples - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        return clip_id, waveform


dataset = AudioDataset(valid_clips)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
print(f"Dataset: {len(dataset)} clips | Batches: {len(dataloader)}")

Dataset: 1885 clips | Batches: 118


In [17]:
# Cell 8: Embedding Extraction

if EMBEDDINGS_PATH.exists() and ID_MAP_PATH.exists():
    print("Loading cached embeddings...")
    all_embeddings = np.load(EMBEDDINGS_PATH)
    with open(ID_MAP_PATH, "r") as f:
        id_map = json.load(f)
    print(f"Loaded {all_embeddings.shape[0]} embeddings of dim {all_embeddings.shape[1]}")
else:
    print("Extracting embeddings with BEATs...")
    all_embeddings = []
    id_map = []

    with torch.no_grad():
        for batch_ids, batch_waveforms in tqdm(dataloader, desc="Embedding batches"):
            batch_waveforms = batch_waveforms.to(DEVICE)
            # BEATs expects padding_mask: True where padded
            padding_mask = torch.zeros_like(batch_waveforms, dtype=torch.bool)

            features = beats_model.extract_features(batch_waveforms, padding_mask=padding_mask)[0]
            # features shape: (batch, time_frames, embed_dim) - mean pool over time
            pooled = features.mean(dim=1)  # (batch, embed_dim)
            all_embeddings.append(pooled.cpu().numpy())
            id_map.extend(list(batch_ids))

    all_embeddings = np.vstack(all_embeddings).astype(np.float32)
    print(f"Extracted {all_embeddings.shape[0]} embeddings of dim {all_embeddings.shape[1]}")

    # Cache to disk
    np.save(EMBEDDINGS_PATH, all_embeddings)
    with open(ID_MAP_PATH, "w") as f:
        json.dump(id_map, f)
    print(f"Saved to {EMBEDDINGS_PATH} and {ID_MAP_PATH}")

Extracting embeddings with BEATs...


Embedding batches:   0%|          | 0/118 [00:00<?, ?it/s]

Extracted 1885 embeddings of dim 768
Saved to embeddings.npy and id_map.json


In [18]:
# Cell 9: Build Similarity Index

# L2-normalize for cosine similarity
norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)
embeddings_normed = (all_embeddings / norms).astype(np.float32)

if USE_FAISS:
    dim = embeddings_normed.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings_normed)
    print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")
else:
    index = NearestNeighbors(n_neighbors=11, metric="cosine", algorithm="brute")
    index.fit(embeddings_normed)
    print(f"sklearn NearestNeighbors fitted: {embeddings_normed.shape[0]} vectors")

FAISS index built: 1885 vectors, dim=768


In [ ]:
# Cell 10: Recommendation Function

id_to_idx = {cid: i for i, cid in enumerate(id_map)}

def recommend(clip_id: str, k: int = 10) -> pd.DataFrame:
    """Return top-k most similar clips for a given clip_id."""
    if clip_id not in id_to_idx:
        raise ValueError(f"clip_id '{clip_id}' not found in index.")

    idx = id_to_idx[clip_id]
    query_vec = embeddings_normed[idx:idx+1]

    if USE_FAISS:
        scores, indices = index.search(query_vec, k + 1)
        scores = scores[0]
        indices = indices[0]
    else:
        distances, indices = index.kneighbors(query_vec, n_neighbors=k + 1)
        scores = 1 - distances[0]  # cosine distance -> similarity
        indices = indices[0]

    results = []
    for score, neighbor_idx in zip(scores, indices):
        neighbor_id = id_map[neighbor_idx]
        if neighbor_id == clip_id:
            continue
        results.append({"id": neighbor_id, "similarity": float(score)})
        if len(results) >= k:
            break

    results_df = pd.DataFrame(results)
    # Merge metadata from original dataframe
    meta_cols = ["id", "duration", "speech_ratio", "total_duration_seconds"]
    available_cols = [c for c in meta_cols if c in df.columns]
    results_df = results_df.merge(df[available_cols], on="id", how="left")
    return results_df

In [20]:
# Cell 11: Sanity-Check Evaluation

sample_ids = id_map[:5]  # pick first 5 clips

for sample_id in sample_ids:
    print(f"\n{'='*60}")
    print(f"Query clip: {sample_id}")
    print(f"{'='*60}")
    recs = recommend(sample_id, k=5)
    print(recs.to_string(index=False))

: 

In [ ]:
# Cell 12: Persist Artifacts

# Embeddings and ID map already saved during extraction.
# Save the fitted index.

if USE_FAISS:
    faiss.write_index(index, str(INDEX_PATH.with_suffix(".faiss")))
    print(f"FAISS index saved to {INDEX_PATH.with_suffix('.faiss')}")
else:
    with open(INDEX_PATH, "wb") as f:
        pickle.dump(index, f)
    print(f"sklearn index saved to {INDEX_PATH}")

print(f"\nArtifacts:")
for p in [EMBEDDINGS_PATH, ID_MAP_PATH, INDEX_PATH if not USE_FAISS else INDEX_PATH.with_suffix(".faiss")]:
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p}: {size_mb:.2f} MB")